# Capstone companion --- Chapter 5: Tools as Typed Actions

Chapter~5 argues that an agent's actions must be typed: each tool declares a schema for its input and its output, so a malformed call is rejected at the boundary rather than after it has acted. This companion reads that principle on the capstone banking complaint agent, whose five tools are each defined as a typed action over a Pydantic input and output model.

The five tools are `classify_complaint`, `extract_facts`, `search_policy`, `flag_regulatory` and `draft_response`. Each carries an input model and an output model; the harness validates a proposed call against the input model before the tool runs, and validates the return against the output model before any downstream tool consumes it.

In [ ]:
from agentlab.capstone import banking_tools
from agentlab.capstone.banking_tools import (
    ClassifyInput, ClassifyOutput,
    ExtractInput, ExtractOutput,
    FlagInput, FlagOutput,
    DraftInput, DraftOutput,
)

## The input and output schema of one tool

The classifier maps a customer message to one of three labels. Its typed contract is the pair of models below: the input names the single `message` field, the output names the `category` label and a confidence. Reading the JSON schema shows exactly what the harness checks.

In [ ]:
import json
print('ClassifyInput  :', json.dumps(ClassifyInput.model_json_schema()['properties'], indent=2))
print('ClassifyOutput :', json.dumps(ClassifyOutput.model_json_schema()['properties'], indent=2))

## Assembling the typed toolset

`register_all` binds the five tools into a registry the agent draws from. The policy-search tool is constructed against the policy directory, because its retrieval reads the deployed store; the other four are module-level tool objects. The registry is the typed action space the agent may propose from.

In [ ]:
from agentlab.tools import ToolRegistry
from pathlib import Path

root = Path('.') if Path('data').exists() else Path('..')
registry = ToolRegistry()
banking_tools.register_all(registry, policies_dir=root / 'data' / 'policies')
for name in ('classify_complaint', 'extract_facts', 'search_policy',
             'flag_regulatory', 'draft_response'):
    tool = registry.get(name)
    print(f'{tool.name:20s} in -> {tool.input_schema.__name__:16s} '
          f'out -> {tool.output_schema.__name__:16s} risk={tool.risk}')

## A malformed call is rejected at the boundary

The point of a typed action is that an ill-formed proposal never reaches the tool body. Constructing an input model with a missing or wrong-typed field raises a validation error, so the harness can refuse the call before it acts.

In [ ]:
from pydantic import ValidationError

ok = ClassifyInput(message='I was charged a $35 overdraft fee I did not authorize.')
print('valid input :', ok)
try:
    ClassifyInput()          # missing the required message field
except ValidationError as e:
    print('rejected    :', e.errors()[0]['type'], '-', e.errors()[0]['loc'])

This is the capstone's realization of Chapter~5: the agent's action space is a registry of typed tools, and every call is checked against a declared schema on the way in and on the way out. Chapter~6 adds the execution guard that runs these validated calls safely, and the capstone chapter (Chapter~15) assembles the five tools into the governed workflow.